In [1]:
import operator
from googleapiclient.discovery import build
from dotenv import load_dotenv
import os
import re

In [2]:
load_dotenv(override=True)
API_KEY = os.getenv("YOUTUBE_API_KEY")

In [3]:

youtube = build("youtube", "v3", developerKey=API_KEY)
query = "Advanced Agentic AI Techniques and Applications"

request = youtube.search().list(
    q=query,
    part="snippet",
    maxResults=5,
    type="video"
)

response = request.execute()

video_ids_dict = {}
for item in response["items"]:
    video_ids_dict[item["id"]["videoId"]] = {
        "title": item["snippet"]["title"],
        "description": item["snippet"]["description"],
        "channelID": item["snippet"]["channelId"],
        "channelTitle": item["snippet"]["channelTitle"],
        "url": item["snippet"]["thumbnails"]["high"]["url"],
    }

TimeoutError: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond

In [ ]:
def get_counts(video_id):
    request = youtube.videos().list(
        part="snippet,statistics,contentDetails",
        id=video_id
    )

    response = request.execute()

    video = response["items"][0]
    views = int(video["statistics"]["viewCount"])
    likes = int(video["statistics"]["likeCount"])
    comments = int(video["statistics"]["commentCount"])
    # print("Title:", video["snippet"]["title"])
    # print("Views:", video["statistics"].get("viewCount"))
    # print("Likes:", video["statistics"].get("likeCount"))
    # print("Comments Count:", video["statistics"].get("commentCount"))
    # print("Duration:", video["contentDetails"]["duration"])

    return {"views" : views, "likes" : likes, "comments" : comments}



In [ ]:
def get_authority_score(video_id, video_ids_dict):
    max_views= max(v['views'] for v in video_ids_dict.values())
    min_views= min(v['views'] for v in video_ids_dict.values())
    max_likes= max(v['likes'] for v in video_ids_dict.values())
    min_likes= min(v['likes'] for v in video_ids_dict.values())
    max_comments= max(v['comments'] for v in video_ids_dict.values())
    min_comments= min(v['comments'] for v in video_ids_dict.values())

    current_views = video_ids_dict[video_id]['views']
    current_likes = video_ids_dict[video_id]['likes']
    current_comments = video_ids_dict[video_id]['comments']
    normalized_views = (current_views - min_views) / (max_views - min_views)
    normalized_likes = (current_likes - min_likes) / (max_likes - min_likes)
    normalized_comments = (current_comments - min_comments) / (max_comments - min_comments)
    authority_score = 0.5 * normalized_views + 0.3 * normalized_likes + 0.2 * normalized_comments
    return authority_score



In [ ]:
counts_dict = {}
for video_id in video_ids_dict.keys():
    counts_dict[video_id] = get_counts(video_id)

authority_score_dict = {}
for video_id in video_ids_dict.keys():
    authority_score_dict[video_id] = round(get_authority_score(video_id, counts_dict),2)

sorted_videos_based_on_authority_score = dict(sorted(authority_score_dict.items(), key=operator.itemgetter(1), reverse=True))
print(sorted_videos_based_on_authority_score)





In [ ]:
def get_video_summaries(sorted_videos_based_on_authority_score):
    video_summaries = []
    for video_id in sorted_videos_based_on_authority_score:
        score = sorted_videos_based_on_authority_score[video_id]
        if score >= 0.8:
            recommendation = "recommended as core curriculum material"
        elif score >= 0.6:
            recommendation = "recommended as primary supporting material"
        elif score >= 0.4:
            recommendation = "suitable as supplementary learning"
        else:
            recommendation = "optional reference material"
        text = f"""
        The video "{video_ids_dict[video_id]['title']}" published by
        {video_ids_dict[video_id]['channelTitle']}
        achieves an authority score of {score:.2f}.

        Based on engagement and quality metrics, it is {recommendation}
        for learners exploring this topic.
        """
        video_summaries.append(text)
    return video_summaries



In [ ]:
print([re.sub(r"\n|\s{2,}", " ", video_summary).strip() for video_summary in get_video_summaries(sorted_videos_based_on_authority_score)])